In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Localizar a planilha
candidates = [
    Path("data/raw/conversations.xlsx"),
    Path("../data/raw/conversations.xlsx")
]

DATA_PATH = next(
    (path.resolve() for path in candidates if path.exists()),
    None
)

print("Diretório atual:", Path.cwd())
print("Planilha encontrada:", DATA_PATH)

Diretório atual: c:\Users\corac\OneDrive\Área de Trabalho\claudia-ai-operations-case\notebooks
Planilha encontrada: C:\Users\corac\OneDrive\Área de Trabalho\claudia-ai-operations-case\data\raw\conversations.xlsx


In [2]:
# Carregar todas as abas do arquivo Excel
sheets = pd.read_excel(DATA_PATH, sheet_name=None)

# Mostrar os nomes das abas
print("Abas encontradas:")
for sheet in sheets.keys():
    print("-", sheet)

Abas encontradas:
- CASE
- links
- conversations & responses


In [3]:
# Selecionar a aba principal
df = sheets["conversations & responses"].copy()

# Verificar dimensões da base
print("Quantidade de linhas:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])

Quantidade de linhas: 2858
Quantidade de colunas: 13


In [4]:
# Listar todas as colunas da base
print("Colunas disponíveis:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Colunas disponíveis:

1. conversation.cloudchatid
2. conversation.problemsolved
3. conversation.isn2
4. conversation.tag
5. conversation.createdat
6. conversation.resolutionreason
7. response.userquery
8. response.rawfinalanswer
9. response.modelreasoning
10. section.title
11. section.response
12. section.type
13. section.flowid


In [5]:
# Informações gerais da base
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2858 entries, 0 to 2857
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   conversation.cloudchatid       2858 non-null   str           
 1   conversation.problemsolved     147 non-null    float64       
 2   conversation.isn2              1396 non-null   float64       
 3   conversation.tag               2858 non-null   str           
 4   conversation.createdat         2858 non-null   datetime64[us]
 5   conversation.resolutionreason  2858 non-null   str           
 6   response.userquery             2483 non-null   str           
 7   response.rawfinalanswer        2852 non-null   str           
 8   response.modelreasoning        2407 non-null   str           
 9   section.title                  1374 non-null   str           
 10  section.response               1374 non-null   str           
 11  section.type                

In [6]:
print("VALORES DE PROBLEMSOLVED:")
print(df["conversation.problemsolved"].value_counts(dropna=False))

print("\nVALORES DE ISN2:")
print(df["conversation.isn2"].value_counts(dropna=False))

VALORES DE PROBLEMSOLVED:
conversation.problemsolved
NaN    2711
1.0     101
0.0      46
Name: count, dtype: int64

VALORES DE ISN2:
conversation.isn2
NaN    1462
1.0    1396
Name: count, dtype: int64


In [7]:
# Identificar os motivos de resolução registrados
print(
    df["conversation.resolutionreason"]
    .value_counts(dropna=False)
    .to_string()
)

conversation.resolutionreason
Tickets N1                                                                      1475
Cliente Pediu Humano                                                             334
Eddie Transferiu                                                                 279
Claudia usou conteúdo N2                                                         262
Interação superior ao limite de mensagens                                        198
Resolved in N2. Reason: Limit of consecutive clarification attempts reached       80
Claudia pediu detalhes e não entendeu a solicitação do cliente                    65
Claudia não resolveu o problema do cliente                                        46
Resolved in N2. Reason: No relevant content found to answer customer request      44
Claudia não sentiu confiança na resposta                                          28
Erro na chamada do Eddie                                                          25
Resolved in N2. Reason: Limit of re

In [8]:
# Quantidade de conversas únicas
total_conversas = df["conversation.cloudchatid"].nunique()

# Quantidade de motivos diferentes por conversa
motivos_por_conversa = (
    df.groupby("conversation.cloudchatid")[
        "conversation.resolutionreason"
    ].nunique()
)

print("Conversas únicas:", total_conversas)

print(
    "\nConversas com mais de um motivo:",
    (motivos_por_conversa > 1).sum()
)

print(
    "\nMaior quantidade de motivos em uma conversa:",
    motivos_por_conversa.max()
)

Conversas únicas: 580

Conversas com mais de um motivo: 0

Maior quantidade de motivos em uma conversa: 1


In [9]:
# Quantidade de registros por conversa
registros_por_conversa = (
    df.groupby("conversation.cloudchatid")
    .size()
)

print("Estatísticas de registros por conversa:")
print(registros_por_conversa.describe())

Estatísticas de registros por conversa:
count    580.000000
mean       4.927586
std        3.041157
min        1.000000
25%        3.000000
50%        4.000000
75%        6.000000
max       21.000000
dtype: float64


In [10]:
# Identificar as 10 conversas mais longas
top_conversas = (
    df.groupby("conversation.cloudchatid")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print(top_conversas.to_string())

conversation.cloudchatid
281e6f69-7d5b-4b71-8348-67fefc620871    21
58ab0e25-b9d5-4d55-a0de-734020dde8ae    18
843ffd5e-f156-44b8-80b6-dd8d4803fe72    18
a80ec12f-a54c-44ff-9c0b-2cabd813aeef    17
5446fd4d-b8c7-41d5-9d33-9143e878be6e    16
43423df6-bd32-4ce4-a581-9c5b7ae7cbfe    15
f426637b-9ea4-4c93-9c89-216fe4d341d5    15
678debff-97db-42b4-ba72-860f612ca33b    14
b9db2bfc-d0f3-433c-8421-93016b6d8b36    14
1cfedceb-7f52-45d5-8eab-f5d30e5829f2    14


In [11]:
# Criar tabela com quantidade de registros por conversa
volume_conversas = (
    df.groupby("conversation.cloudchatid")
    .size()
    .reset_index(name="total_registros")
)

# Adicionar o motivo de resolução
motivos = (
    df.groupby("conversation.cloudchatid")[
        "conversation.resolutionreason"
    ].first()
    .reset_index()
)

# Juntar as informações
analise_conversas = volume_conversas.merge(
    motivos,
    on="conversation.cloudchatid"
)

# Mostrar as 10 conversas mais longas
top_10 = analise_conversas.sort_values(
    "total_registros",
    ascending=False
).head(10)

display(top_10)

,conversation.cloudchatid,total_registros,conversation.resolutionreason
99,281e6f69-7d5b-4b71-8348-67fefc620871,21,Interação superior ao limite de mensagens
225,58ab0e25-b9d5-4d55-a0de-734020dde8ae,18,Eddie Transferiu
322,843ffd5e-f156-44b8-80b6-dd8d4803fe72,18,Interação superior ao limite de mensagens
411,a80ec12f-a54c-44ff-9c0b-2cabd813aeef,17,Interação superior ao limite de mensagens
215,5446fd4d-b8c7-41d5-9d33-9143e878be6e,16,Tickets N1
178,43423df6-bd32-4ce4-a581-9c5b7ae7cbfe,15,Interação superior ao limite de mensagens
560,f426637b-9ea4-4c93-9c89-216fe4d341d5,15,Interação superior ao limite de mensagens
261,678debff-97db-42b4-ba72-860f612ca33b,14,Tickets N1
451,b9db2bfc-d0f3-433c-8421-93016b6d8b36,14,Interação superior ao limite de mensagens
74,1cfedceb-7f52-45d5-8eab-f5d30e5829f2,14,Interação superior ao limite de mensagens


In [12]:
# Selecionar a conversa mais longa
id_conversa = top_10.iloc[0]["conversation.cloudchatid"]

# Filtrar todos os registros dessa conversa
conversa = df[
    df["conversation.cloudchatid"] == id_conversa
]

# Mostrar as informações relevantes
colunas = [
    "response.userquery",
    "response.rawfinalanswer",
    "conversation.resolutionreason",
    "section.title",
    "section.response"
]

display(conversa[colunas].to_string(index=False))

'                                                                        response.userquery                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           response.rawfinalanswer             conversation.resolutionreason                                                                                                                                                                                                                                                                                 

In [13]:
# Identificar conversas encerradas por excesso de mensagens
conversas_limite = (
    df[
        df["conversation.resolutionreason"]
        == "Interação superior ao limite de mensagens"
    ]
    ["conversation.cloudchatid"]
    .drop_duplicates()
)

print("Conversas que atingiram o limite:")
print(len(conversas_limite))

Conversas que atingiram o limite:
14


In [15]:
# Filtrar registros das conversas que atingiram o limite
df_limite = df[
    df["conversation.cloudchatid"].isin(conversas_limite)
]

# Identificar os assuntos registrados
assuntos = (
    df_limite[
        ["conversation.cloudchatid", "conversation.tag"]
    ]
    .drop_duplicates()
    .sort_values("conversation.tag")
)

display(assuntos.to_string(index=False))

'            conversation.cloudchatid                                  conversation.tag\nb9db2bfc-d0f3-433c-8421-93016b6d8b36                                 DFes-Manifestação\nf426637b-9ea4-4c93-9c89-216fe4d341d5                       apps-relatóriosavançadosnfe\n843ffd5e-f156-44b8-80b6-dd8d4803fe72                       configurações-certificadoa3\n1cfedceb-7f52-45d5-8eab-f5d30e5829f2                            configurações-usuários\na80ec12f-a54c-44ff-9c0b-2cabd813aeef                                  dfes-consultanfe\n639bd8de-d6f5-4efe-af2f-f5b11a294653                                  dfes-consultanfe\nca98e767-2055-4d81-a7ef-517f74c09e58                                 dfes-consultanfse\n9eb9b6e6-9137-4c9a-b7a7-de549bde5048                                dfes-notasemresumo\n43423df6-bd32-4ce4-a581-9c5b7ae7cbfe                                dfes-notasemresumo\n281e6f69-7d5b-4b71-8348-67fefc620871 eddie_administrativos-alteração/exclusãodeusuário\n9b53ca45-dde6-4a3d-ad98-9f531a7

In [18]:
ids_migracao = df_limite[
    df_limite["conversation.tag"]
    .str.contains("migração", case=False, na=False)
]["conversation.cloudchatid"].unique()

for id_conversa in ids_migracao:
    print("\n" + "=" * 80)
    print("CONVERSA:", id_conversa)

    conversa = df_limite[
        df_limite["conversation.cloudchatid"] == id_conversa
    ]

    display(
        conversa[
            [
                "response.userquery",
                "response.rawfinalanswer",
                "conversation.resolutionreason"
            ]
        ].to_string(index=False)
    )


CONVERSA: 9b53ca45-dde6-4a3d-ad98-9f531a71e9d7


'                                                          response.userquery                                                                                                                                                                                                                                         response.rawfinalanswer             conversation.resolutionreason\n                                          COMO FAÇO PARA EXCLUIR UMA EMPRESA                                                                                                         Vejo que você deseja excluir uma empresa, você também pode inativar o CNPJ ou CPF, para isso é necessário apenas um perfil de administrador ou gerente. Interação superior ao limite de mensagens\n                                        EXCLUIR EMPRESAS QUE EU NAO USO MAIS                                                                                                         Vejo que você deseja excluir uma empresa, você também pode inativa


CONVERSA: 99f445fe-6a02-4792-a4e2-f298343fb8de


'                                                          response.userquery                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       response.rawfinalanswer             conversation.resolutionreason\n                                                          preciso de suporte                                                                                                                                                                                                                                                                                                    Verifiquei por aq


CONVERSA: 9c66ae82-0e04-4b6e-bc07-a40a4cda4d8c


'                                                                    response.userquery                                                                                                                                                                                                                                                                                                                                                                                                  response.rawfinalanswer             conversation.resolutionreason\no cnpj [ID] esta vinculado a uma conta do [COMPANY] mais preciso vincular a esta conta                                                                                                                                                            Quando a informação "Já existe uma empresa com CNPJ/CPF" aparece, significa que este CNPJs já está cadastrado em uma conta [COMPANY].\\n\\nPara verificar se já adicionou à sua conta basta verificar a seção "Empresas da

In [19]:
for id_conversa in ids_migracao:
    conversa = df_limite[
        df_limite["conversation.cloudchatid"] == id_conversa
    ]

    print("\n" + "=" * 80)
    print("CONVERSA:", id_conversa)
    print("Quantidade de linhas:", len(conversa))

    print("\nDados preenchidos por coluna:")
    display(conversa.notna().sum())

    print("\nTodos os dados da conversa:")
    display(conversa.to_string(index=False))


CONVERSA: 9b53ca45-dde6-4a3d-ad98-9f531a71e9d7
Quantidade de linhas: 11

Dados preenchidos por coluna:


conversation.cloudchatid         11
conversation.problemsolved        0
conversation.isn2                11
conversation.tag                 11
conversation.createdat           11
conversation.resolutionreason    11
response.userquery                5
response.rawfinalanswer          11
response.modelreasoning           4
section.title                     2
section.response                  2
section.type                      2
section.flowid                    2
dtype: int64


Todos os dados da conversa:


'            conversation.cloudchatid  conversation.problemsolved  conversation.isn2                     conversation.tag  conversation.createdat             conversation.resolutionreason                                                           response.userquery                                                                                                                                                                                                                                         response.rawfinalanswer                                                                                                                                                                                                                                                            response.modelreasoning                                                                                                                                                                                                             


CONVERSA: 99f445fe-6a02-4792-a4e2-f298343fb8de
Quantidade de linhas: 13

Dados preenchidos por coluna:


conversation.cloudchatid         13
conversation.problemsolved        0
conversation.isn2                13
conversation.tag                 13
conversation.createdat           13
conversation.resolutionreason    13
response.userquery               10
response.rawfinalanswer          13
response.modelreasoning          10
section.title                     2
section.response                  2
section.type                      2
section.flowid                    2
dtype: int64

... (2 outputs left out)


CONVERSA: 9c66ae82-0e04-4b6e-bc07-a40a4cda4d8c
Quantidade de linhas: 12

Dados preenchidos por coluna:


conversation.cloudchatid         12
conversation.problemsolved        0
conversation.isn2                12
conversation.tag                 12
conversation.createdat           12
conversation.resolutionreason    12
response.userquery                8
response.rawfinalanswer          12
response.modelreasoning           8
section.title                     3
section.response                  3
section.type                      3
section.flowid                    3
dtype: int64


Todos os dados da conversa:


'            conversation.cloudchatid  conversation.problemsolved  conversation.isn2                     conversation.tag  conversation.createdat             conversation.resolutionreason                                                                     response.userquery                                                                                                                                                                                                                                                                                                                                                                                                  response.rawfinalanswer                                                                                                                                                                                                                                                                                                                             

In [20]:
ids_notas_resumo = df_limite[
    df_limite["conversation.tag"]
    .str.contains("notasemresumo", case=False, na=False)
]["conversation.cloudchatid"].unique()

for id_conversa in ids_notas_resumo:
    conversa = df_limite[
        df_limite["conversation.cloudchatid"] == id_conversa
    ]

    print("\n" + "=" * 80)
    print("CONVERSA:", id_conversa)

    display(
        conversa[
            [
                "response.userquery",
                "response.rawfinalanswer",
                "conversation.resolutionreason"
            ]
        ].to_string(index=False)
    )


CONVERSA: 9eb9b6e6-9137-4c9a-b7a7-de549bde5048


"                                                                                                                                                                                                                    response.userquery                                                                                                                                                                                                                                                                                                                                                                                                                                                                    response.rawfinalanswer             conversation.resolutionreason\n                                                                                                                                                                                                     de 180 dias\\neu posso fazer isso?                   


CONVERSA: 43423df6-bd32-4ce4-a581-9c5b7ae7cbfe


'                                                                                                                                                    response.userquery                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        response.rawfinalanswer             conversation.resolutionreason\ntem várias notas que foram emitidas em 30/06 que está como somente resumo, não consigo baixar o XML. E

In [21]:
for id_conversa in ids_notas_resumo:
    conversa = df_limite[
        df_limite["conversation.cloudchatid"] == id_conversa
    ]

    print("\n" + "=" * 80)
    print("ID:", id_conversa)
    print("Linhas encontradas:", len(conversa))

    for _, linha in conversa.iterrows():
        pergunta = linha["response.userquery"]
        resposta = linha["response.rawfinalanswer"]

        if pd.notna(pergunta) or pd.notna(resposta):
            print("\nCLIENTE:", pergunta)
            print("CLAUDIA:", resposta)


ID: 9eb9b6e6-9137-4c9a-b7a7-de549bde5048
Linhas encontradas: 14

CLIENTE: de 180 dias
eu posso fazer isso?
CLAUDIA: Sim, você pode realizar a manifestação 'Operação não Realizada' dentro do prazo de 180 dias após a emissão da nota. Essa opção é válida para informar à Sefaz que a operação foi acordada, mas não foi concluída. Caso precise de mais apoio, é só avisar. 🙂

CLIENTE: de 180 dias
eu posso fazer isso?
CLAUDIA: Sim, você pode realizar a manifestação 'Operação não Realizada' dentro do prazo de 180 dias após a emissão da nota. Essa opção é válida para informar à Sefaz que a operação foi acordada, mas não foi concluída. Caso precise de mais apoio, é só avisar. 🙂

CLIENTE: pdf
pdf
CLAUDIA: Entendido! Mesmo que você tenha o PDF da nota fiscal, se ela estiver em 'somente resumo' na plataforma, você pode realizar uma manifestação conclusiva, como 'Operação não Realizada', para indicar que a mercadoria não foi entregue. Isso pode ajudar a liberar o XML completo na próxima consulta, desd

In [22]:
for id_conversa in ids_notas_resumo:
    conversa = df_limite[
        df_limite["conversation.cloudchatid"] == id_conversa
    ]

    arquivo = (
        conversa[
            ["response.userquery", "response.rawfinalanswer"]
        ]
        .to_string(index=False)
    )

    nome = f"conversa_{id_conversa[:8]}.txt"

    Path("outputs").mkdir(exist_ok=True)
    Path("outputs", nome).write_text(arquivo, encoding="utf-8")

    print("Arquivo salvo:", nome)

Arquivo salvo: conversa_9eb9b6e6.txt
Arquivo salvo: conversa_43423df6.txt


In [23]:
# Uma linha por conversa, evitando contar várias interações como atendimentos diferentes
df_conversas = df.drop_duplicates(
    subset="conversation.cloudchatid"
).copy()

# Contagem dos motivos de encerramento
motivos = (
    df_conversas["conversation.resolutionreason"]
    .value_counts(dropna=False)
    .reset_index()
)

motivos.columns = ["motivo", "quantidade"]

motivos["percentual"] = (
    motivos["quantidade"] / len(df_conversas) * 100
).round(2)

display(motivos)

,motivo,quantidade,percentual
0,Tickets N1,337,58.10
1,Claudia usou conteúdo N2,69,11.90
2,Cliente Pediu Humano,63,10.86
3,Eddie Transferiu,49,8.45
4,Interação superior ao limite de mensagens,14,2.41
5,Resolved in N2. Reason: Limit of consecutive c...,12,2.07
6,Claudia pediu detalhes e não entendeu a solici...,10,1.72
7,Claudia não resolveu o problema do cliente,7,1.21
8,Resolved in N2. Reason: No relevant content fo...,6,1.03
9,Erro na chamada do Eddie,5,0.86


In [24]:
df_conversas[
    df_conversas["conversation.resolutionreason"] == "Tickets N1"
][
    [
        "conversation.cloudchatid",
        "conversation.tag",
        "conversation.problemsolved",
        "conversation.isn2",
        "conversation.resolutionreason"
    ]
].head(15)

,conversation.cloudchatid,conversation.tag,conversation.problemsolved,conversation.isn2,conversation.resolutionreason
2,ab653864-6038-4721-96e1-78207e9d5f0b,Configurações-CertificadoA1,NaN,NaN,Tickets N1
5,4a666fa5-d1b9-45f0-98a5-c23867c581d8,configurações-certificadoa1,NaN,NaN,Tickets N1
7,d0c9a478-51c6-4449-b6de-eeb2f6741685,configurações-certificadoa1,NaN,NaN,Tickets N1
13,1cde4a94-491e-4193-8dd6-8bce89e28cde,eddie_financeiro-nfse[EMPRESA],NaN,NaN,Tickets N1
18,363fd4bc-e9e4-4b35-8573-d6cd6b8d5c03,configurações-certificadoa1,NaN,NaN,Tickets N1
23,12004cb5-17b1-4541-abf4-5b420a04566c,eddie_financeiro-boleto,NaN,NaN,Tickets N1
24,3a754c16-a37e-4f7f-bc02-4b6a25f17337,eddie_financeiro-boleto,NaN,NaN,Tickets N1
28,45356a4e-e2bd-40be-800b-447719fb1632,dfes-recuperaçãodenotas,NaN,NaN,Tickets N1
30,8878cf86-a0bf-45b1-b27e-d3da180ce511,dfes-consultanfe,NaN,NaN,Tickets N1
33,6fda2f83-d203-4ba6-bd85-fd1f72e6f153,eddie_financeiro-nfse[EMPRESA],NaN,NaN,Tickets N1


In [25]:
# Selecionar 3 conversas classificadas como Tickets N1
ids_n1 = df_conversas[
    df_conversas["conversation.resolutionreason"] == "Tickets N1"
]["conversation.cloudchatid"].head(3)

# Exibir as mensagens dessas conversas
amostra_n1 = df[
    df["conversation.cloudchatid"].isin(ids_n1)
][
    [
        "conversation.cloudchatid",
        "response.userquery",
        "response.rawfinalanswer"
    ]
]

display(amostra_n1.to_string(index=False))

'            conversation.cloudchatid                                                                                                                                                                    response.userquery                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                response.rawfinalanswer\nab653864-6038-4721-96e1-78207e9d5f0b Bom dia, [NAME]... ref. ao CNPJ [ANONYMIZED] nã

In [26]:
tags = (
    df_conversas["conversation.tag"]
    .value_counts()
    .head(15)
    .reset_index()
)

tags.columns = ["tag", "quantidade"]

tags["percentual"] = (
    tags["quantidade"] / len(df_conversas) * 100
).round(2)

display(tags)

,tag,quantidade,percentual
0,dfes-notasemresumo,63,10.86
1,dfes-consultanfe,56,9.66
2,eddie_administrativos-migraçãodecnpj,42,7.24
3,eddie_financeiro-nfse[EMPRESA],35,6.03
4,outros-instabilidade/bug[EMPRESA],33,5.69
5,outros-outromotivo,28,4.83
6,configurações-certificadoa1,25,4.31
7,configurações-dfelist,25,4.31
8,financeiro-liberar/bloquearacesso,19,3.28
9,eddie_financeiro-boleto,17,2.93


In [27]:
cruzamento = pd.crosstab(
    df_conversas["conversation.tag"],
    df_conversas["conversation.resolutionreason"]
)

cruzamento["total"] = cruzamento.sum(axis=1)

cruzamento = cruzamento.sort_values(
    "total", ascending=False
)

display(cruzamento.head(15))

conversation.resolutionreason,Claudia detectou ação de transferência,Claudia não resolveu o problema do cliente,Claudia não sentiu confiança na resposta,Claudia pediu detalhes e não entendeu a solicitação do cliente,Claudia usou conteúdo N2,Cliente Pediu Humano,Eddie Transferiu,Erro na chamada do Eddie,Interação superior ao limite de mensagens,Resolved in N2. Reason: Limit of consecutive clarification attempts reached,Resolved in N2. Reason: Limit of repeated section usage reached,Resolved in N2. Reason: No relevant content found to answer customer request,Tickets N1,total
conversation.tag,,,,,,,,,,,,,,
dfes-notasemresumo,1,1,0,1,0,1,0,0,2,1,1,1,54,63
dfes-consultanfe,0,1,2,2,0,10,0,0,2,2,0,0,37,56
eddie_administrativos-migraçãodecnpj,0,0,0,1,0,3,27,0,3,0,0,0,8,42
eddie_financeiro-nfse[EMPRESA],0,0,0,0,0,0,4,3,1,0,0,0,27,35
outros-instabilidade/bug[EMPRESA],0,0,0,0,3,3,9,0,0,0,0,0,18,33
outros-outromotivo,0,0,0,0,19,6,0,0,0,1,0,0,2,28
configurações-certificadoa1,0,0,0,0,0,1,0,0,0,1,0,1,22,25
configurações-dfelist,0,0,0,0,0,1,0,0,0,0,0,1,23,25
financeiro-liberar/bloquearacesso,0,0,0,0,19,0,0,0,0,0,0,0,0,19


In [28]:
ids_migracao = df_conversas[
    (df_conversas["conversation.tag"] == "eddie_administrativos-migraçãodecnpj") &
    (df_conversas["conversation.resolutionreason"] == "Eddie Transferiu")
]["conversation.cloudchatid"]

amostra_migracao = df[
    df["conversation.cloudchatid"].isin(ids_migracao)
][
    [
        "conversation.cloudchatid",
        "response.userquery",
        "response.rawfinalanswer"
    ]
]

display(amostra_migracao.to_string(index=False))

'            conversation.cloudchatid                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       response.userquery                                                                                                          

# 02. Diagnóstico executivo

## 2.1 Visão geral da operação

A base analisada contém 2.858 registros, correspondentes a 580 conversas únicas.

Como a planilha resulta da junção entre conversas, respostas e seções utilizadas, uma mesma conversa pode aparecer em várias linhas. Por isso, os indicadores de volume e encerramento foram calculados considerando uma ocorrência por conversa.

## 2.2 Principais assuntos dos atendimentos

Os assuntos mais frequentes identificados foram:

- Notas em resumo: 63 conversas (10,86%).
- Consulta NF-e: 56 (9,66%).
- Migração de CNPJ: 42 (7,24%).
- NFSe empresarial: 35 (6,03%).
- Instabilidade ou bug: 33 (5,69%).

As quatro categorias relacionadas a DF-e — notas em resumo, consulta NF-e, consulta NFSe e lista de DF-e — somam 160 conversas, equivalentes a 27,59% da base.

## 2.3 Motivos de encerramento

Os principais motivos registrados foram:

- Tickets N1: 337 conversas (58,10%).
- Claudia usou conteúdo N2: 69 (11,90%).
- Cliente pediu humano: 63 (10,86%).
- Eddie Transferiu: 49 (8,45%).

Essas categorias representam motivos de encerramento e não devem ser interpretadas automaticamente como sucesso ou falha da Claudia.

## 2.4 Oportunidade identificada: migração de CNPJ

Foram identificadas 42 conversas com a tag de migração de CNPJ. Dessas, 27 terminaram com o motivo "Eddie Transferiu", correspondendo a 64,29% dos atendimentos dessa categoria.

A análise qualitativa identificou situações como:
- CNPJ já vinculado a outra conta;
- Tentativa de cadastrar novamente um CNPJ anteriormente utilizado;
- Necessidade de autorização do administrador da conta de origem;
- Cliente sem acesso à conta em que o CNPJ está cadastrado.

Esses casos indicam uma oportunidade de melhorar a identificação da intenção, a coleta de informações e a condução do fluxo de migração.

Parte das transferências, entretanto, pode ser necessária por questões de segurança, titularidade e autorização. Portanto, não é correto considerar todas as transferências como evitáveis.

## 2.5 Diagnóstico preliminar

Os dados e as conversas examinadas apontam oportunidades em três frentes:

1. Melhorar a identificação da intenção do cliente e a diferenciação entre solicitações semelhantes.
2. Aprimorar a condução dos fluxos, evitando perguntas repetidas e orientações que não consideram informações já fornecidas.
3. Direcionar solicitações especializadas para agentes com escopo, ferramentas e critérios claros de transferência.

A migração de CNPJ é uma oportunidade candidata para um Specialist Agent, devido à concentração de transferências observada e à existência de um fluxo operacional específico.

## 2.6 Limitações da análise

A base contém campos incompletos e registros de diferentes etapas do atendimento. Além disso, o motivo "Tickets N1" ainda não foi validado como indicador de resolução autônoma.

Assim, os resultados apresentados não permitem, isoladamente, afirmar a taxa atual de resolução autônoma nem calcular a distância exata até a meta de 75%.